# Published-reference result plot
Plots stored reference numbers, not newly measured model results. Keep this separate from evaluation workflows.


In [ ]:
from amgpn.config import require as _cfg_require, resolve as _cfg_resolve, format_value as _cfg_format
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.style.use('seaborn-v0_8-paper')

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'axes.labelsize': 12,
    'font.size': 12,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': False,
    'figure.dpi': _cfg_require('GPN_V5_test.ipynb.cell1.module.size_or_budget'),
    'savefig.dpi': _cfg_require('GPN_V5_test.ipynb.cell1.module.size_or_budget__2'),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'errorbar.capsize': 4,
    'lines.markeredgewidth': 1.5
})

palette = sns.color_palette("colorblind", 10)

# 固定配色：与图例风格一致
colors = {
    'MN': '#FF9999',        # 粉红
    'ProtoNet': '#43F05A',  # 亮绿色
    'GPN': '#67CFF0',       # 浅蓝
    'AMGPN': '#FFA23A',     # 橙色

    # 新增模型
    'UNEM': '#FF9999',      # 使用 MN 的粉红色
    'FEAT': '#9B59B6',      # 另选紫色，和现有颜色区分明显
}

datasets = {
    'DroneRFb-Spectra': {
        'FEAT': {'acc': [82.2, 88.1, 89.0, 89.3], 'ci': [2.3, 1.4, 1.2, 1.4]},
        'ProtoNet': {'acc': [82.7, 88.9, 91.7, 90.5], 'ci': [2.0, 1.3, 0.9, 0.9]},
        'GPN': {'acc': [81.9, 90.0, 91.3, 92.0], 'ci': [2.0, 1.2, 1.0, 0.8]},
        'AMGPN': {'acc': [81.8, 93.7, 94.9, 96.3], 'ci': [2.0, 0.8, 0.6, 0.4]},
        'UNEM': {'acc': [71.9, 88.0, 91.3, 92.8], 'ci': [2.7, 1.3, 0.9, 0.6]},
    },
    'RFUAV': {
        'FEAT': {'acc': [26.3, 28.1, 27.8, 27.2], 'ci': [1.7, 1.2, 1.4, 1.5]},
        'ProtoNet': {'acc': [56.2, 64.6, 65.9, 66.1], 'ci': [1.9, 1.7, 1.8, 1.7]},
        'GPN': {'acc': [55.5, 66.3, 69.6, 71.4], 'ci': [1.7, 1.6, 1.6, 1.4]},
        'AMGPN': {'acc': [51.7, 74.2, 81.6, 86.1], 'ci': [1.8, 1.6, 1.3, 1.3]},
        'UNEM': {'acc': [44.9, 68.3, 77.1, 83.5], 'ci': [2.3, 1.9, 1.6, 1.4]},
    }
}

shots = _cfg_require('GPN_V5_test.ipynb.cell1.module.shots')
models = ['ProtoNet', 'GPN', 'AMGPN', 'FEAT', 'UNEM']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for idx, (dataset_name, data) in enumerate(datasets.items()):
    ax = axes[idx]

    # 关键修改 <configured>：拉大不同 shot 之间的间距
    x = np.arange(len(shots)) * 1.35

    # 关键修改 <configured>：缩小单根柱宽，避免每组柱子挤满整个 shot 区间
    width = _cfg_require('GPN_V5_test.ipynb.cell1.module.width')

    for i, model in enumerate(models):
        # 关键修改 <configured>：<configured> 个模型时正确居中
        offset = (i - (len(models) - 1) / 2) * width

        ax.bar(
            x + offset,
            data[model]['acc'],
            width,
            yerr=data[model]['ci'],
            capsize=3,
            label=model if idx == 0 else "",
            color=colors[model],
            alpha=_cfg_require('GPN_V5_test.ipynb.cell1.module.alpha__2'),
            edgecolor='black',
            linewidth=0.8
        )

    ax.set_xlabel('K-shot', fontsize=13, fontweight='bold')
    ax.set_ylabel('Accuracy (%)', fontsize=13, fontweight='bold')
    ax.set_title(dataset_name, fontsize=14, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(shots)
    ax.tick_params(labelsize=11)
    ax.grid(axis='y', alpha=_cfg_require('GPN_V5_test.ipynb.cell1.module.alpha'), linestyle='--')

    if dataset_name == 'DroneRFb-Spectra':
        ax.set_ylim(20, 100)
    else:
        # 关键修改 <configured>：RFUAV 的 FEAT 只有 <configured>–<configured>，不能从 <configured> 开始
        ax.set_ylim(20, 90)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.05),
    ncol=5,
    fontsize=12,
    frameon=True,
    edgecolor='black'
)

fig.suptitle(
    'Few-Shot Classification Performance (95% Confidence Intervals)',
    fontsize=15,
    fontweight='bold',
    y=1.12
)

plt.tight_layout()
plt.savefig('comparison_two_datasets.pdf', bbox_inches='tight', dpi=_cfg_require('GPN_V5_test.ipynb.cell1.module.size_or_budget__3'))
plt.show()